# 01 - 布林带因子计算

**功能**: 计算布林带指标并相对化

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from scipy import stats

# 数据库连接
PROJECT_ROOT = Path('../../').resolve()
db_path = PROJECT_ROOT / 'data' / 'invest.db'
conn = sqlite3.connect(db_path)
print("✅ 数据库连接成功")

In [ ]:
def 计算布林带 (close, period=20, num_std=2.0):
    """计算布林带"""
    middle = close.rolling(period).mean()
    std = close.rolling(period).std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    
    # 股价在布林带中的位置（0-1）
    position = (close - lower) / (upper - lower)
    
    # 带宽
    bandwidth = (upper - lower) / middle
    
    return {
        'upper': upper,
        'middle': middle,
        'lower': lower,
        'position': position,
        'bandwidth': bandwidth
    }

def 计算分位数 (series, lookback=250):
    """计算滚动分位数"""
    return series.rolling(lookback).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100 if len(x) > 10 else np.nan
    )

print("✅ 函数定义完成")

In [ ]:
# 读取数据
daily_df = pd.read_sql_query("SELECT * FROM stock_daily ORDER BY trade_date", conn)
daily_df['trade_date'] = pd.to_datetime(daily_df['trade_date'], format='%Y%m%d')

print(f"数据量：{len(daily_df)} 条记录")
print(f"股票数量：{daily_df['ts_code'].nunique()}")

In [ ]:
# 计算单只股票
test_code = daily_df['ts_code'].iloc[0]
stock_data = daily_df[daily_df['ts_code'] == test_code].sort_values('trade_date').copy()
stock_data.set_index('trade_date', inplace=True)

print(f"测试股票：{test_code}")

# 计算布林带
bollinger = 计算布林带 (stock_data['close'])
stock_data['boll_position'] = bollinger['position']
stock_data['boll_bandwidth'] = bollinger['bandwidth']
stock_data['boll_bandwidth_pct'] = 计算分位数 (bollinger['bandwidth'])

print("✅ 布林带计算完成")
print("\n最新数据：")
print(stock_data[['close', 'boll_position', 'boll_bandwidth', 'boll_bandwidth_pct']].tail())

In [ ]:
# 可视化
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# 股价 + 布林带
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['close'], name='收盘价', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=bollinger['upper'], name='上轨', line=dict(color='red', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=bollinger['lower'], name='下轨', line=dict(color='green', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=bollinger['middle'], name='中轨', line=dict(color='orange')), row=1, col=1)

# 布林带位置
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['boll_position'], name='位置', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=0.8, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=0.2, line_dash="dash", line_color="green", row=2, col=1)

fig.update_layout(title=f'{test_code} - 布林带', height=700, hovermode='x unified')
fig.show()